# 01_data_collection_mop.ipynb
## Project Title: Traffic Accident Risk Prediction (TARP)

**Unit:** SIT782  
**Prepared by:** Subathira Thinakaran  

**Project Team:**  
- Suba (225094537)  
- Burhan (224802775)  
- Khalid (224696667)  

**Task:** Data Collection – Import and Explore MOP Pedestrian and Bicycle Datasets (Week 1 of 8)

## Objective
This notebook implements a data collection pipeline using the City of Melbourne Open Data API.  
It retrieves pedestrian and bicycle datasets, validates their structure, and stores them locally for further preprocessing and analysis.

## Datasets Used
- Pedestrian Counting System (Hourly Data)
- Annual Bike Counts (Super Tuesday)

## Output
- pedestrian.csv  
- bicycle.csv  

## API Key Setup

This notebook uses the City of Melbourne Open Data API.

To run this notebook:
1. Open the "Secrets" tab in Google Colab (🔑 icon)
2. Add a new secret:
   - Name: MOP_API_KEY
   - Value: Your API key
3. Run the notebook

Note: API keys are not stored in this notebook for security reasons.

## API Configuration

This section defines:
- API base URL  
- Dataset identifiers  
- Secure API key access  
- Local storage path  

In [1]:
import requests
import pandas as pd
from pathlib import Path
from google.colab import userdata

# -----------------------------
# 1. API configuration
# -----------------------------
BASE_URL = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets"

PEDESTRIAN_DATASET = "pedestrian-counting-system-monthly-counts-per-hour"
BICYCLE_DATASET = "traffic-count-vehicle-classification-2014-2017"

API_KEY = userdata.get("MOP_API_KEY")
if not API_KEY:
    raise ValueError("MOP_API_KEY not found in Colab secrets.")

RAW_DATA_DIR = Path("data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("API base URL:", BASE_URL)
print("Pedestrian dataset:", PEDESTRIAN_DATASET)
print("Bicycle dataset:", BICYCLE_DATASET)
print("API key loaded:", API_KEY is not None)

API base URL: https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets
Pedestrian dataset: pedestrian-counting-system-monthly-counts-per-hour
Bicycle dataset: traffic-count-vehicle-classification-2014-2017
API key loaded: True


## Helper Functions

This section defines reusable functions to interact with the MOP API.  
`fetch_records()` retrieves a single batch of records, while `fetch_all_records()` handles pagination to collect the full dataset efficiently.

In [2]:
# -----------------------------
# 2. Helper functions
# -----------------------------
import time

def fetch_records(dataset_id, limit=100, offset=0, where=None, select=None, order_by=None):
    """
    Fetch one page of records from the City of Melbourne Open Data API.
    """
    headers = {
        "Authorization": f"Apikey {API_KEY}"
    }

    url = f"{BASE_URL}/{dataset_id}/records"
    params = {
        "limit": limit,
        "offset": offset
    }

    if where:
        params["where"] = where
    if select:
        params["select"] = select
    if order_by:
        params["order_by"] = order_by

    try:
        response = requests.get(url, params=params, headers=headers, timeout=60)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from {dataset_id}: {e}")
        return {}


def fetch_all_records(dataset_id, batch_size=100, where=None, select=None, order_by=None, max_records=None):
    """
    Fetch all records from a dataset using pagination.
    """
    all_rows = []
    offset = 0

    while True:
        payload = fetch_records(
            dataset_id=dataset_id,
            limit=batch_size,
            offset=offset,
            where=where,
            select=select,
            order_by=order_by
        )

        rows = payload.get("results", [])
        if not rows:
            break

        all_rows.extend(rows)
        offset += batch_size

        if len(all_rows) % 1000 == 0 or (max_records is not None and len(all_rows) >= max_records):
            print(f"Fetched {len(all_rows)} records from {dataset_id}")

        if max_records is not None and len(all_rows) >= max_records:
            all_rows = all_rows[:max_records]
            break

        time.sleep(0.1)

    return pd.DataFrame(all_rows)

## API Connectivity Test

This step verifies that the API is accessible and returning valid data.  
A small sample of records is retrieved to confirm successful communication and response structure.

In [3]:
# -----------------------------
# 3. Test API connectivity
# -----------------------------
test_json = fetch_records(
    dataset_id=PEDESTRIAN_DATASET,
    limit=5
)

print("Keys in response:", test_json.keys())
print("Number of preview rows:", len(test_json.get("results", [])))

# Preview sample data
pd.DataFrame(test_json["results"]).head()

Keys in response: dict_keys(['total_count', 'results'])
Number of preview rows: 5


,id,location_id,sensing_date,hourday,direction_1,direction_2,pedestriancount,sensor_name,location
0,135920240721,135,2024-07-21,9,226,225,451,Spen161_T,"{'lon': 144.95319102, 'lat': -37.8172861}"
1,135220250609,135,2025-06-09,2,9,21,30,Spen161_T,"{'lon': 144.95319102, 'lat': -37.8172861}"
2,132520250803,132,2025-08-03,5,8,4,12,King335_T,"{'lon': 144.95386444, 'lat': -37.81267639}"
3,1342020240829,134,2024-08-29,20,311,233,544,Spen201_T,"{'lon': 144.95303935, 'lat': -37.81694221}"
4,1342020260104,134,2026-01-04,20,264,253,517,Spen201_T,"{'lon': 144.95303935, 'lat': -37.81694221}"


## Pedestrian Data Preview

Convert the API response into a pandas DataFrame and inspect the structure and sample records of the pedestrian dataset.

In [4]:
# -----------------------------
# 4. Preview pedestrian data
# -----------------------------
pedestrian_preview = pd.DataFrame(test_json["results"])

print("Preview shape:", pedestrian_preview.shape)
print("Columns:", list(pedestrian_preview.columns))

pedestrian_preview.head()

Preview shape: (5, 9)
Columns: ['id', 'location_id', 'sensing_date', 'hourday', 'direction_1', 'direction_2', 'pedestriancount', 'sensor_name', 'location']


,id,location_id,sensing_date,hourday,direction_1,direction_2,pedestriancount,sensor_name,location
0,135920240721,135,2024-07-21,9,226,225,451,Spen161_T,"{'lon': 144.95319102, 'lat': -37.8172861}"
1,135220250609,135,2025-06-09,2,9,21,30,Spen161_T,"{'lon': 144.95319102, 'lat': -37.8172861}"
2,132520250803,132,2025-08-03,5,8,4,12,King335_T,"{'lon': 144.95386444, 'lat': -37.81267639}"
3,1342020240829,134,2024-08-29,20,311,233,544,Spen201_T,"{'lon': 144.95303935, 'lat': -37.81694221}"
4,1342020260104,134,2026-01-04,20,264,253,517,Spen201_T,"{'lon': 144.95303935, 'lat': -37.81694221}"


### Observation

The dataset contains temporal (date and hour), directional counts, and location-based information.  
The `pedestriancount` field represents total pedestrian activity, while the `location` field contains geographic coordinates that can be used for spatial analysis.

## Full Pedestrian Dataset Import

This step retrieves the pedestrian dataset using pagination to handle large volumes of data.  
A subset of records is loaded during development for efficiency.

In [5]:
# -----------------------------
# 5. Import pedestrian dataset
# -----------------------------
# During development, a subset of records is loaded for efficiency.
# The full dataset can be retrieved later by removing max_records.

pedestrian_df = fetch_all_records(
    dataset_id=PEDESTRIAN_DATASET,
    batch_size=100,
    max_records=5000
)

print("Pedestrian dataset shape:", pedestrian_df.shape)
print("Number of unique sensors:", pedestrian_df["sensor_name"].nunique())

pedestrian_df.head()

Fetched 1000 records from pedestrian-counting-system-monthly-counts-per-hour
Fetched 2000 records from pedestrian-counting-system-monthly-counts-per-hour
Fetched 3000 records from pedestrian-counting-system-monthly-counts-per-hour
Fetched 4000 records from pedestrian-counting-system-monthly-counts-per-hour
Fetched 5000 records from pedestrian-counting-system-monthly-counts-per-hour
Pedestrian dataset shape: (5000, 9)
Number of unique sensors: 98


,id,location_id,sensing_date,hourday,direction_1,direction_2,pedestriancount,sensor_name,location
0,135920240721,135,2024-07-21,9,226,225,451,Spen161_T,"{'lon': 144.95319102, 'lat': -37.8172861}"
1,135220250609,135,2025-06-09,2,9,21,30,Spen161_T,"{'lon': 144.95319102, 'lat': -37.8172861}"
2,132520250803,132,2025-08-03,5,8,4,12,King335_T,"{'lon': 144.95386444, 'lat': -37.81267639}"
3,1342020240829,134,2024-08-29,20,311,233,544,Spen201_T,"{'lon': 144.95303935, 'lat': -37.81694221}"
4,1342020260104,134,2026-01-04,20,264,253,517,Spen201_T,"{'lon': 144.95303935, 'lat': -37.81694221}"


### Observation

A total of 5000 pedestrian records were successfully imported from the MOP API.  
The dataset contains 98 unique sensors and includes temporal, directional, and geographic attributes, which will support later cleaning, exploratory analysis, and feature engineering.

## Bicycle Dataset Preview

Retrieve a sample of the bicycle dataset using the MOP API and inspect its structure and attributes.

In [13]:
# -----------------------------
# 6. Preview bicycle dataset
# -----------------------------
bicycle_preview = pd.DataFrame(fetch_records(BICYCLE_DATASET, limit=5)["results"])

print("Bicycle preview shape:", bicycle_preview.shape)
print("Total columns:", len(bicycle_preview.columns))
print("Columns:", list(bicycle_preview.columns))

key_columns = ["date", "time", "road_name", "suburb", "direction", "bike"]
print("\nKey columns for project use:", key_columns)

bicycle_preview.head()

Bicycle preview shape: (5, 28)
Total columns: 28
Columns: ['date', 'road_name', 'location', 'suburb', 'speed_limit', 'direction', 'time', 'vehicle_class_1', 'vehicle_class_2', 'vehicle_class_3', 'vehicle_class_4', 'vehicle_class_5', 'vehicle_class_6', 'vehicle_class_7', 'vehicle_class_8', 'vehicle_class_9', 'vehicle_class_10', 'vehicle_class_11', 'vehicle_class_12', 'vehicle_class_13', 'motorcycle', 'bike', 'average_speed', '85th_percentile_speed', 'maximum_speed', 'road_segment', 'road_segment_1', 'road_segment_2']

Key columns for project use: ['date', 'time', 'road_name', 'suburb', 'direction', 'bike']


,date,road_name,location,suburb,speed_limit,direction,time,vehicle_class_1,vehicle_class_2,vehicle_class_3,...,vehicle_class_12,vehicle_class_13,motorcycle,bike,average_speed,85th_percentile_speed,maximum_speed,road_segment,road_segment_1,road_segment_2
0,2015-04-02,Rathdowne Street,Between Kay Street and Palmerston Street,Carlton,60,N,12:00,658,1,84,...,0,0,None,None,45.0,54.0,None,20567,20568,20569
1,2015-05-02,Rathdowne Street,Between Kay Street and Palmerston Street,Carlton,60,N,1:00,88,0,0,...,0,0,None,None,50.2,58.0,None,20567,20568,20569
2,2015-05-02,Rathdowne Street,Between Kay Street and Palmerston Street,Carlton,60,N,7:00,351,0,36,...,0,0,None,None,46.3,57.0,None,20567,20568,20569
3,2015-05-02,Rathdowne Street,Between Kay Street and Palmerston Street,Carlton,60,N,10:00,545,1,70,...,0,0,None,None,44.8,54.9,None,20567,20568,20569
4,2015-05-02,Rathdowne Street,Between Kay Street and Palmerston Street,Carlton,60,N,20:00,659,2,27,...,0,1,None,None,43.4,53.0,None,20567,20568,20569


### Observation

The bicycle dataset contains date and time fields, road and suburb information, direction, and a dedicated `bike` count column. This makes it more suitable for the TARP project than the earlier annual snapshot dataset, as it supports time-based mobility analysis.

However, the preview also shows that the `bike` field may contain missing values, which will need to be addressed during the data cleaning stage.

## Full Bicycle Dataset Import

This step retrieves the bicycle dataset using the MOP API.  
Since the dataset is relatively small, all available records are loaded using pagination.

In [14]:
# -----------------------------
# 7. Import bicycle dataset
# -----------------------------
bicycle_df = fetch_all_records(
    dataset_id=BICYCLE_DATASET,
    batch_size=100,
    max_records=5000
)

print("Bicycle dataset shape:", bicycle_df.shape)
print("Number of unique roads:", bicycle_df["road_name"].nunique())

# Check availability of bike data
missing_bike = bicycle_df["bike"].isna().sum()
print("Missing bike values:", missing_bike)
print("Available bike records:", len(bicycle_df) - missing_bike)

bicycle_df.head()

Fetched 1000 records from traffic-count-vehicle-classification-2014-2017
Fetched 2000 records from traffic-count-vehicle-classification-2014-2017
Fetched 3000 records from traffic-count-vehicle-classification-2014-2017
Fetched 4000 records from traffic-count-vehicle-classification-2014-2017
Fetched 5000 records from traffic-count-vehicle-classification-2014-2017
Bicycle dataset shape: (5000, 28)
Number of unique roads: 87
Missing bike values: 1891
Available bike records: 3109


,date,road_name,location,suburb,speed_limit,direction,time,vehicle_class_1,vehicle_class_2,vehicle_class_3,...,vehicle_class_12,vehicle_class_13,motorcycle,bike,average_speed,85th_percentile_speed,maximum_speed,road_segment,road_segment_1,road_segment_2
0,2015-04-02,Rathdowne Street,Between Kay Street and Palmerston Street,Carlton,60,N,12:00,658.0,1.0,84.0,...,0.0,0.0,NaN,NaN,45.0,54.0,None,20567,20568,20569
1,2015-05-02,Rathdowne Street,Between Kay Street and Palmerston Street,Carlton,60,N,1:00,88.0,0.0,0.0,...,0.0,0.0,NaN,NaN,50.2,58.0,None,20567,20568,20569
2,2015-05-02,Rathdowne Street,Between Kay Street and Palmerston Street,Carlton,60,N,7:00,351.0,0.0,36.0,...,0.0,0.0,NaN,NaN,46.3,57.0,None,20567,20568,20569
3,2015-05-02,Rathdowne Street,Between Kay Street and Palmerston Street,Carlton,60,N,10:00,545.0,1.0,70.0,...,0.0,0.0,NaN,NaN,44.8,54.9,None,20567,20568,20569
4,2015-05-02,Rathdowne Street,Between Kay Street and Palmerston Street,Carlton,60,N,20:00,659.0,2.0,27.0,...,0.0,1.0,NaN,NaN,43.4,53.0,None,20567,20568,20569


### Observation

A total of 5000 bicycle-related traffic records were successfully imported from the City of Melbourne Open Data API, covering 87 unique road locations.

The dataset includes date and time fields, enabling time-based analysis. However, the `bike` column contains missing values, indicating that not all traffic surveys recorded bicycle counts.

This will require filtering or handling of missing values during the data cleaning stage to ensure accurate bicycle traffic analysis.

## Basic Validation

This step validates the structure of both datasets by inspecting their shapes and column names.  
It also highlights differences in dataset size and complexity.

In [17]:
# -----------------------------
# 8. Basic validation
# -----------------------------
print("Pedestrian dataset shape:", pedestrian_df.shape)
print("Number of pedestrian columns:", len(pedestrian_df.columns))

print("\nBicycle dataset shape:", bicycle_df.shape)
print("Number of bicycle columns:", len(bicycle_df.columns))

print("\nPedestrian columns:")
for col in pedestrian_df.columns:
    print("-", col)

print("\nBicycle columns:")
for col in bicycle_df.columns:
    print("-", col)

Pedestrian dataset shape: (5000, 9)
Number of pedestrian columns: 9

Bicycle dataset shape: (5000, 28)
Number of bicycle columns: 28

Pedestrian columns:
- id
- location_id
- sensing_date
- hourday
- direction_1
- direction_2
- pedestriancount
- sensor_name
- location

Bicycle columns:
- date
- road_name
- location
- suburb
- speed_limit
- direction
- time
- vehicle_class_1
- vehicle_class_2
- vehicle_class_3
- vehicle_class_4
- vehicle_class_5
- vehicle_class_6
- vehicle_class_7
- vehicle_class_8
- vehicle_class_9
- vehicle_class_10
- vehicle_class_11
- vehicle_class_12
- vehicle_class_13
- motorcycle
- bike
- average_speed
- 85th_percentile_speed
- maximum_speed
- road_segment
- road_segment_1
- road_segment_2


### Observation

The pedestrian dataset has a simpler structure, with only 9 columns focused on pedestrian counts and mobility timing. In contrast, the bicycle dataset contains 28 columns, including date, time, road context, direction, speed-related variables, and a bike count field.

This indicates that the bicycle dataset provides richer contextual traffic information, but it will require more selective cleaning and feature selection before analysis.

## Missing Values Analysis

This step checks for missing values in both datasets to identify data quality issues and determine necessary cleaning steps.

In [19]:
# -----------------------------
# 9. Missing values check
# -----------------------------
print("Missing values in pedestrian dataset:")
print(pedestrian_df.isna().sum())

print("\nTop missing values in bicycle dataset:")
missing_bicycle = bicycle_df.isna().sum().sort_values(ascending=False)
print(missing_bicycle.head(10))

empty_columns = missing_bicycle[missing_bicycle == len(bicycle_df)]
print(f"\nNumber of completely empty columns in bicycle dataset: {len(empty_columns)}")

# Focus on bike column (important for project)
missing_bike = bicycle_df["bike"].isna().sum()
print(f"\nMissing 'bike' values: {missing_bike}")
print(f"Available 'bike' records: {len(bicycle_df) - missing_bike}")

Missing values in pedestrian dataset:
id                 0
location_id        0
sensing_date       0
hourday            0
direction_1        0
direction_2        0
pedestriancount    0
sensor_name        0
location           0
dtype: int64

Top missing values in bicycle dataset:
road_segment_2           4765
road_segment_1           4592
motorcycle               1891
bike                     1891
maximum_speed            1855
average_speed             482
85th_percentile_speed     482
vehicle_class_5             7
vehicle_class_2             7
vehicle_class_3             7
dtype: int64

Number of completely empty columns in bicycle dataset: 0

Missing 'bike' values: 1891
Available 'bike' records: 3109


### Observation

The pedestrian dataset does not contain any missing values, indicating high data quality and minimal preprocessing requirements.

In contrast, the bicycle dataset contains several missing values across multiple columns. Notably, the `bike` column has a significant number of missing values, indicating that bicycle counts were not recorded in all traffic surveys.

However, no columns are completely empty, suggesting that useful information is still available. The missing values in the `bike` column will need to be handled carefully during the data cleaning stage, likely by filtering records with valid bicycle counts for analysis.

## Save Raw Datasets

This step saves the collected datasets locally as CSV files for further preprocessing and analysis in subsequent stages.

In [21]:
# -----------------------------
# 10. Save raw datasets
# -----------------------------
pedestrian_file = RAW_DATA_DIR / "pedestrian.csv"
bicycle_file = RAW_DATA_DIR / "bicycle.csv"

pedestrian_df.to_csv(pedestrian_file, index=False)
bicycle_df.to_csv(bicycle_file, index=False)

print("Pedestrian data saved to:", pedestrian_file)
print("Number of pedestrian records:", len(pedestrian_df))

print("\nBicycle data saved to:", bicycle_file)
print("Number of bicycle records:", len(bicycle_df))

print("\nFile existence check:")
print("Pedestrian file exists:", pedestrian_file.exists())
print("Bicycle file exists:", bicycle_file.exists())

Pedestrian data saved to: data/raw/pedestrian.csv
Number of pedestrian records: 5000

Bicycle data saved to: data/raw/bicycle.csv
Number of bicycle records: 5000

File existence check:
Pedestrian file exists: True
Bicycle file exists: True


### Observation

Both pedestrian and bicycle datasets have been successfully saved as raw CSV files in the designated directory. The saved files preserve the original structure of the datasets and will be used in the next stage for data cleaning and transformation.

## Final Summary

This section provides a summary of the data collection process, including the number of records retrieved and the file locations where the datasets have been stored.

In [24]:
# -----------------------------
# 11. Final summary
# -----------------------------

from google.colab import files

summary = {
    "Pedestrian records": len(pedestrian_df),
    "Bicycle records": len(bicycle_df),
    "Pedestrian file": str(pedestrian_file),
    "Bicycle file": str(bicycle_file),
}

for key, value in summary.items():
    print(f"{key}: {value}")

files.download(str(pedestrian_file))
files.download(str(bicycle_file))

Pedestrian records: 5000
Bicycle records: 5000
Pedestrian file: data/raw/pedestrian.csv
Bicycle file: data/raw/bicycle.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Conclusion

The data collection process has been successfully completed using the City of Melbourne Open Data API.  

Both pedestrian and bicycle datasets were retrieved, validated, and stored locally.  
The pipeline is reproducible and ready for the next stage, which involves data cleaning and preprocessing.